# Lab 02: How Spark plans a query

**Time:** about 2 h 30 min.

**Checklist questions this lab gives you evidence for:**<br><span style="color:#d93025"><b>1.6 Catalyst Optimizer</b></span><br><span style="color:#d93025"><b>1.8 Tungsten</b></span><br><span style="color:#d93025"><b>1.13 DataFrame vs. RDD</b></span>

**Goal:** read query plans, watch Catalyst rewrite a query, and see what a Python UDF costs. The last section is DataFrame aggregation practice.

## Data information
A small online shop with **one table**. The setup cell below creates it in `data/orders.parquet` (1,000,000 orders from January to March 2024; seeded, so everyone gets the same rows):
* *order_id*; *customer_id* (empty for guest checkout), *customer_name*, *country*, *segment* (`retail`, `business` or `guest`)
* *product_id*, *category* (8 categories), *price* (price of one item)
* *quantity* (1 to 5) and *amount* = quantity × price, in dollars
* *status*: `paid`, `shipped`, `cancelled` or `returned`; *payment_method*: `card`, `cash` or `wallet`
* *order_ts* and *order_date*: when the order was placed; *tracking_code*: a 32-character code

Customer 1 is "Big Reseller Ltd": a business customer who orders in bulk (quantity 5) and places 30% of all orders. Some labs make a small lookup table from this one table (for example one row per customer); the notebook says when.

## How to work
* Write your answer in the code cell under each question and run it. The answers are at the very end of this notebook (collapsed): open one only after you tried. Each answer shows the code and the result of the reference run, so you can compare your numbers.
* When a question says *guess first*, write your guess as a comment before you run anything.
* Keep the Spark UI open (the setup cell prints its address).
* Stuck for more than 10 minutes? Read the documentation list at the top, then open the answer of that question.


## Documentation
Everything you need is on these pages. The questions do not repeat them: read, try things out in a cell, and find the answers yourself.

* [DataFrame.groupBy](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.groupBy.html)
* [DataFrame.orderBy](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.orderBy.html)
* [SELECT reference](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select.html)
* [Chapter 6: Old SQL, New Tricks](https://spark.apache.org/docs/latest/api/python/user_guide/sql.html)
* [GroupedData.agg](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.GroupedData.agg.html)
* [DataFrame.join](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.join.html)
* [EXPLAIN](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-explain.html)
* [Catalyst Optimizer (blog)](https://www.databricks.com/blog/2015/04/13/deep-dive-into-spark-sqls-catalyst-optimizer.html)
* [Optimizer rules (Japila)](https://books.japila.pl/spark-sql-internals/logical-optimizations/)
* [Spark SQL configuration](https://spark.apache.org/docs/latest/configuration.html#runtime-sql-configuration)
* [EnsureRequirements (Japila)](https://books.japila.pl/spark-sql-internals/physical-optimizations/EnsureRequirements/)
* [functions.udf](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.udf.html)
* [Parquet filter pushdown](https://spark.apache.org/docs/latest/sql-data-sources-parquet.html#configuration)
* [Parquet files](https://spark.apache.org/docs/latest/sql-data-sources-parquet.html)
* [Project Tungsten (blog)](https://www.databricks.com/blog/2015/04/28/project-tungsten-bringing-spark-closer-to-bare-metal.html)
* [RDD.reduceByKey](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.RDD.reduceByKey.html)
* [functions.weekday](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.weekday.html)
* [ANALYZE TABLE](https://spark.apache.org/docs/latest/sql-ref-syntax-aux-analyze-table.html)
* [Leveraging statistics](https://spark.apache.org/docs/latest/sql-performance-tuning.html#leveraging-statistics)
* [functions.when](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.when.html)
* [Chapter 3: Function Junction](https://spark.apache.org/docs/latest/api/python/user_guide/dataprep.html)
* [GroupedData.pivot](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.GroupedData.pivot.html)
* [DataFrame.filter](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.filter.html)

## Setup
Run the cell below once (its code is hidden: click it to expand). It creates the data and starts Spark.

In [ ]:
# Setup: run this cell once. It creates the shop data in data/ and starts Spark.
# Nothing to solve here, and you do not need to read it.
import contextlib, hashlib, io, json, os, re, shutil, sys, time, urllib.request
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # Spark's Python workers use this kernel's Python
from pyspark.sql import SparkSession, Window, functions as F

# ---------------------------------------------------------------- the shop data ---------
# A small online shop with ONE table, generated from seeded hash functions: every machine gets the same rows.
COUNTRIES = ["Uzbekistan", "Germany", "USA", "India", "Brazil", "Japan"]
CATEGORIES = ["books", "toys", "garden", "sports", "kitchen", "beauty", "music", "office"]
N_CUSTOMERS, N_PRODUCTS, N_ORDERS = 10_000, 200, 1_000_000

def _u(col, salt):
    """Pseudo-random number in [0, 1) computed from a column (same result on every machine)."""
    return F.pmod(F.xxhash64(col, F.lit(salt)), F.lit(1_000_000)) / 1_000_000.0

def _pick(u, options):
    return F.element_at(F.array(*[F.lit(o) for o in options]), (F.floor(u * len(options)) + 1).cast("int"))

def make_orders(spark, n=N_ORDERS, dirty=False):
    i = F.col("id")
    cust = (F.when(_u(i, 11) < 0.02, F.lit(None).cast("long"))                        # guest checkout: no customer
             .when(_u(i, 10) < 0.30, F.lit(1).cast("long"))                           # the big reseller: 30% of orders
             .otherwise((F.floor(_u(i, 12) * (N_CUSTOMERS - 1)) + 2).cast("long")))
    name = F.when(cust == 1, "Big Reseller Ltd").when(cust.isNotNull(), F.concat(F.lit("Customer "), cust.cast("string")))
    country = _pick(F.when(cust.isNotNull(), _u(cust, 1)).otherwise(_u(i, 31)), COUNTRIES)
    segment = F.when(cust.isNull(), "guest").when((cust == 1) | (_u(cust, 3) < 0.2), "business").otherwise("retail")
    product = (F.floor(_u(i, 14) * N_PRODUCTS) + 1).cast("int")
    category = F.element_at(F.array(*[F.lit(c) for c in CATEGORIES]), (product % 8 + 1).cast("int"))
    price = F.round((product * 37) % 195 + 5.99, 2)
    quantity = F.when(cust == 1, F.lit(5)).otherwise(F.floor(_u(i, 13) * 5) + 1).cast("int")   # the reseller always orders in bulk
    ts = F.lit("2024-01-01 00:00:00").cast("timestamp") + F.make_interval(secs=F.floor(_u(i, 15) * 91 * 86400).cast("int"))
    u = _u(i, 17)
    status = F.when(u < 0.55, "paid").when(u < 0.80, "shipped").when(u < 0.92, "cancelled").otherwise("returned")
    v = _u(i, 18)
    payment = F.when(v < 0.60, "card").when(v < 0.85, "cash").otherwise("wallet")
    df = spark.range(0, n, 1, 4).select(
        (i + 1).alias("order_id"), cust.alias("customer_id"), name.alias("customer_name"), country.alias("country"), segment.alias("segment"),
        product.alias("product_id"), category.alias("category"), price.alias("price"), quantity.alias("quantity"),
        F.round(quantity * price, 2).alias("amount"), status.alias("status"), payment.alias("payment_method"),
        ts.alias("order_ts"), F.md5(i.cast("string")).alias("tracking_code"))
    df = df.withColumn("order_date", F.to_date("order_ts"))
    if dirty:    # the monthly files of the final project contain mistakes on purpose
        df = df.withColumn("amount", F.when(_u(F.col("order_id"), 19) < 0.008, -F.col("amount")).otherwise(F.col("amount")))
        df = df.withColumn("quantity", F.when(_u(F.col("order_id"), 20) < 0.006, 0).otherwise(F.col("quantity")))
        df = df.unionByName(df.filter(_u(F.col("order_id"), 21) < 0.005))              # exact duplicates
    return df.select("order_id", "customer_id", "customer_name", "country", "segment", "product_id", "category", "price", "quantity",
                     "amount", "status", "payment_method", "order_ts", "order_date", "tracking_code")

def _create_data(spark):
    os.makedirs("data/monthly", exist_ok=True)
    make_orders(spark).repartition(4).write.mode("overwrite").parquet("data/orders.parquet")
    dirty = make_orders(spark, dirty=True)
    wrong = _u(F.col("order_id"), 22) < 0.003          # a few orders land in the wrong month's file
    month = F.month("order_date")
    file_month = F.when(wrong, month % 3 + 1).otherwise(month)
    for m in (1, 2, 3):
        (dirty.filter(file_month == m).coalesce(1).write.mode("overwrite")
              .parquet(f"data/monthly/orders_2024-{m:02d}.parquet"))

# ---------------------------------------------------------------- the Spark session ----
def new_session(master="local[4]", driver_memory="2g", delta=False, avro=False, aqe=False, conf=None):
    builder = (SparkSession.builder.master(master).appName("day02_lab")
               .config("spark.driver.memory", driver_memory)
               .config("spark.ui.showConsoleProgress", "false")
               .config("spark.sql.adaptive.enabled", str(aqe).lower())   # off = exact job and stage counts
               .config("spark.sql.session.timeZone", "UTC")                # same dates and hours for everyone
               .config("spark.sql.warehouse.dir", os.path.abspath("spark-warehouse"))
               # Spark 4.2.0 bug (SPARK-60294): Python UDFs on Parquet can hang without this.
               .config("spark.sql.execution.arrow.pythonUDF.columnarInput.enabled", "false"))
    for k, v in (conf or {}).items():
        builder = builder.config(k, v)
    if delta:
        from delta import configure_spark_with_delta_pip
        builder = (builder.config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
                          .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
        builder = configure_spark_with_delta_pip(builder, extra_packages=["org.apache.spark:spark-avro_2.13:4.2.0"] if avro else None)
    elif avro:
        builder = builder.config("spark.jars.packages", "org.apache.spark:spark-avro_2.13:4.2.0")
    s = builder.getOrCreate()
    s.sparkContext.setLogLevel("ERROR")
    return s

for _d in ("spark-warehouse", "metastore_db"):
    shutil.rmtree(_d, ignore_errors=True)               # a clean catalog on every start
spark = new_session()
def _data_ok():
    """True if the single-table shop data is already there (an older three-table version is rebuilt)."""
    try:
        return "customer_name" in spark.read.parquet("data/orders.parquet").columns and os.path.exists("data/monthly/orders_2024-03.parquet")
    except Exception:
        return False

if not _data_ok():
    print("creating the shop data (about 30 seconds, once)...")
    _create_data(spark)
print(f"Spark {spark.version} | UI: {spark.sparkContext.uiWebUrl}")

# ---------------------------------------------------------------- helpers ----------------
def _tracker():
    return spark.sparkContext.statusTracker()

def _run_group(action):
    sc = spark.sparkContext
    group = f"g{time.time_ns()}"
    sc.setJobGroup(group, group)
    try:
        action()
    finally:
        sc.setLocalProperty("spark.jobGroup.id", None)
    return [_tracker().getJobInfo(j) for j in sorted(_tracker().getJobIdsForGroup(group))]

def jobs_run(action):
    """How many Spark jobs calling action() starts."""
    return len(_run_group(action))

def stages_run(action):
    """How many stages those jobs had in total."""
    return sum(len(j.stageIds) for j in _run_group(action))

def last_stage_tasks(action):
    """Number of tasks in the last stage action() ran."""
    jobs = _run_group(action)
    return _tracker().getStageInfo(max(s for j in jobs for s in j.stageIds)).numTasks

def _api(path):
    sc = spark.sparkContext
    return json.load(urllib.request.urlopen(f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}{path}"))

api = _api   # read anything from the Spark UI's REST API, e.g. api("/executors")

def _stage_rows(action):
    jobs = _run_group(action)
    ids = {s for j in jobs for s in j.stageIds}
    return [s for s in _api("/stages?length=2000") if s["stageId"] in ids]

def shuffle_bytes(action):
    """Total shuffle bytes written by the stages action() ran."""
    return sum(s["shuffleWriteBytes"] for s in _stage_rows(action))

def spill_bytes(action):
    """(memory, disk) bytes spilled by the stages action() ran."""
    rows = _stage_rows(action)
    return sum(s["memoryBytesSpilled"] for s in rows), sum(s["diskBytesSpilled"] for s in rows)

def skew_ratio(action):
    """Slowest task / median task, in the stage with the most tasks that action() ran."""
    rows = _stage_rows(action)
    st = max(rows, key=lambda s: s["numTasks"])
    tasks = _api(f"/stages/{st['stageId']}/{st['attemptId']}/taskList?length=5000")
    times = sorted(t["taskMetrics"]["executorRunTime"] for t in tasks if t.get("taskMetrics"))
    return round(times[-1] / max(times[len(times) // 2], 1), 1)

def failed_attempts():
    """Failed task attempts in the most recent failed stage (from the Spark UI's REST API)."""
    failed = _api("/stages?status=failed")
    if not failed:
        return 0
    st = max(failed, key=lambda s: s["stageId"])
    tasks = _api(f"/stages/{st['stageId']}/{st['attemptId']}/taskList?length=500")
    return sum(1 for t in tasks if t["status"] == "FAILED")

def sql_metric(action, metric):
    """Value of one scan metric (for example 'number of files read') of the query action() runs."""
    desc = f"q{time.time_ns()}"
    spark.sparkContext.setJobDescription(desc)
    try:
        action()
    finally:
        spark.sparkContext.setJobDescription(None)
    for ex in _api("/sql?details=true&planDescription=false&length=2000"):
        if ex.get("description") == desc:
            vals = [m["value"] for n in ex["nodes"] for m in n["metrics"] if m["name"] == metric]
            return vals[0] if vals else None

def noop(df):
    """Run the whole query but throw the result away."""
    df.write.format("noop").mode("overwrite").save()

def plan(df, mode="formatted"):
    """explain() output as a string."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain(mode)
    return buf.getvalue()

def count_nodes(df, *names):
    """How often each operator name appears in the simple plan."""
    text = plan(df, "simple")
    return {n: len(re.findall(r"\b" + re.escape(n) + r"\b", text)) for n in names}

def optimized(df):
    """The optimized logical plan as a string."""
    return df._jdf.queryExecution().optimizedPlan().toString()

def operators(df):
    """Operator names of the physical plan tree, top to bottom."""
    tree = plan(df).split("\n\n")[0].splitlines()[1:]
    return [re.sub(r"\s*\(\d+\)$", "", line).strip(" +-*:") for line in tree]

def rules_fired(df):
    """Names of the Catalyst rules that changed this query's plan."""
    qe = df._jdf.queryExecution(); qe.executedPlan()
    it, fired = qe.tracker().rules().iterator(), set()
    while it.hasNext():
        kv = it.next()
        if kv._2().numEffectiveInvocations() > 0:
            fired.add(kv._1().split(".")[-1].split("$")[-1])
    return fired

def estimate(df):
    """(sizeInBytes, rowCount or None) the optimizer estimates for df."""
    stats = df._jdf.queryExecution().optimizedPlan().stats()
    return int(str(stats.sizeInBytes())), (int(str(stats.rowCount().get())) if stats.rowCount().isDefined() else None)

def partition_sizes(df):
    """Rows per partition, largest first."""
    return sorted((r["count"] for r in df.groupBy(F.spark_partition_id().alias("p")).count().collect()), reverse=True)

def timeit(action):
    start = time.perf_counter(); action(); return round(time.perf_counter() - start, 2)

def folder_mb(path):
    files = [os.path.join(d, f) for d, _, fs in os.walk(path) for f in fs] if os.path.isdir(path) else [path]
    return round(sum(os.path.getsize(f) for f in files) / 1e6, 1)


Load the data the way Lab 01 taught you (given):

In [ ]:
orders = spark.read.parquet("data/orders.parquet")

# a small lookup table made from the same table: one row per customer (given)
(orders.filter(F.col("customer_id").isNotNull())
       .select("customer_id", F.col("country").alias("home_country"), F.col("segment").alias("customer_segment")).distinct()
       .write.mode("overwrite").parquet("data/customers_lookup.parquet"))
customers = spark.read.parquet("data/customers_lookup.parquet")
orders.createOrReplaceTempView("orders")          # so SQL can use the tables too
customers.createOrReplaceTempView("customers")

## A. Warm-up (from Lab 01)

**2.1 How many cancelled orders are there per *payment_method*?** *cancelled_by_payment* = list of `(payment_method, count)` sorted by *payment_method*. Guess how many stages the query has, then measure it into *cancel_stages*.

## B. One question, two syntaxes

**2.2 With SQL: which 3 product categories have the highest average *amount* of paid orders with a *quantity* of at least 2?**
* use `spark.sql` on the view *orders*
* round the average to 2 decimals (*avg_amount*); order by it, ties by *category*
* keep the DataFrame in *sql_df* and the rows as `(category, avg_amount)` tuples in *sql_rows*

**2.3 Answer the same question with the DataFrame API (*api_df*, rows in *api_rows*). Do both versions have the same physical plan?** Compare them with `operators()` and put True or False in *same_plan*.

## C. Read the plan

**2.4 Read the four plans of *q* (`q.explain("extended")`) and answer from them:**
* which filters are pushed down to the *orders* scan? Copy the *PushedFilters* into *pushed* (a list of strings)
* which table does Spark broadcast, `"orders"` or `"customers"`? → *built*
* how many shuffle `Exchange` nodes does the plan have (not `BroadcastExchange`)? → *n_shuffles*

In [ ]:
# given
q = (orders.join(customers, "customer_id")
       .filter(F.col("amount") > 500)
       .groupBy("home_country").count())
q.explain("extended")



## D. Watch Catalyst decide

**2.5 Which Catalyst rules changed the plan of *wasteful*, and which rule removed the silly `1 + 1 == 2`?** Put the set of rule names in *fired* (use `rules_fired()`) and the name of the folding rule in *folding_rule*.

In [ ]:
# given
wasteful = (orders.select("*")
            .join(customers, "customer_id")
            .filter((F.col("amount") > 500) & (F.lit(1) + F.lit(1) == 2))
            .select("home_country", "amount"))



**2.6 Turn off the rule `PushDownPredicates` (the setting `spark.sql.optimizer.excludedRules` takes the full class name `org.apache.spark.sql.catalyst.optimizer.PushDownPredicates`) and build *wasteful* again** (plans are cached per DataFrame). **Is the `amount > 500` filter now above the join?** Look at `optimized(wasteful2)`, answer True or False in *filter_above_join*, then turn the setting off again with `spark.conf.unset`.

**2.7 When does Spark add a shuffle for a `groupBy`?** Count the shuffle `Exchange` nodes (`count_nodes(df, "Exchange")`) of three queries and put them in *exchanges* as `[a, b, c]`:
* a: `orders.groupBy("customer_id").count()`
* b: `orders.repartition("customer_id").groupBy("customer_id").count()`
* c: `orders.repartition(8).groupBy("customer_id").count()`

Guess first.

## E. Pushdown, pruning and the UDF wall

**2.8 Filter *orders* to *amount* over 500 twice: with a normal column condition and with a Python UDF. What does each scan push down, and which is faster?**
* *pushed_builtin* and *pushed_udf*: the *PushedFilters* of each scan (`[]` if there are none)
* time both `count()` calls with `timeit` (*t_builtin*, *t_udf*)

In [ ]:
from pyspark.sql.functions import udf
over_500 = udf(lambda a: a is not None and a > 500, "boolean")   # given


**2.9 Which columns does Spark read from the Parquet files for `orders.agg(F.sum("amount"), F.sum("quantity"))`?** Copy them from *ReadSchema* in the plan into *read_cols* (column names only, sorted).

## F. Tungsten: whole-stage code generation

**2.10 How many whole-stage codegen regions `*(n)` does the simple plan of *cancel_q* have, and which operator separates the regions?** Put the number in *n_codegen* and the operator name in *separator*.

## G. DataFrame vs. RDD

**2.11 How many orders were placed on each weekday (0 = Monday to 6 = Sunday)?** Count them twice, with the RDD API (`orders.rdd.map(...).reduceByKey(...)`, Python's `date.weekday()`) and with the DataFrame API (`F.weekday`), into dicts `{weekday: count}` (*rdd_counts*, *df_counts*), and time both (*t_rdd*, *t_df*). Which is faster?

## H. Statistics change the estimate

**2.12 Save *orders* as the table `orders_t`. How well does the optimizer estimate the number of orders with *amount* over 900?** Use `estimate()` (returns `(sizeInBytes, rowCount)`) and rebuild the DataFrame each time:
* *est_none*: with no statistics
* *est_uniform*: after `ANALYZE TABLE orders_t COMPUTE STATISTICS FOR COLUMNS amount` with `spark.sql.cbo.enabled` on
* *est_histogram*: after turning on `spark.sql.statistics.histogram.enabled` and analyzing again

Put the real count in *actual*. Guess first: will the plain statistics be close?

## I. Coding practice: aggregation

Three problems with `groupBy`, conditional aggregation, `pivot` and filtering on aggregates. Write them from scratch.

**2.13 For every *payment_method*, report the number of orders (*orders*), the share of *cancelled* orders in percent (*cancel_pct*, 1 decimal) and the average *amount* of *paid* orders (*avg_paid*, 2 decimals).** Use one `groupBy` with conditional aggregation (`F.sum(F.when(...))`). *by_payment* = list of `(payment_method, orders, cancel_pct, avg_paid)` sorted by *payment_method*.

**2.14 How many orders of each *status* were placed in each month?** Make a table with one row per month (1 to 3) and one column per status: `month`, `cancelled`, `paid`, `returned`, `shipped`. Use `pivot` with an explicit list of values. *monthly* = list of tuples sorted by month.

**2.15 Which customers placed at least 100 orders?** Ignore guest orders. Put the number of such customers in *n_heavy* and the 3 with the most orders (ties by *customer_id*) in *top3* as `[(customer_id, orders), ...]`.

---
Done? In the Theory part of Day 2, answer the red checklist topics aloud, using the plans you just read as evidence.

## Answers
Try every question yourself first. When you are stuck, click the arrow of that question below to open its answer, then compare it with your own. Each answer is the reference solution with the result of its run; copy the code into the cell above to run it.

<details>
<summary><b>2.1</b> &nbsp;How many cancelled orders are there per payment_method?</summary>

```python
cancel_q = orders.filter(F.col("status") == "cancelled").groupBy("payment_method").count().orderBy("payment_method")
cancelled_by_payment = [tuple(r) for r in cancel_q.collect()]
cancel_stages = stages_run(cancel_q.collect)
cancelled_by_payment, cancel_stages
```

Result:

```
([('card', 72267), ('cash', 29952), ('wallet', 17763)], 3)
```

</details>

<details>
<summary><b>2.2</b> &nbsp;With SQL: which 3 product categories have the highest average amount of paid orders with a ...</summary>

```python
sql_df = spark.sql("""
    SELECT category, round(avg(amount), 2) AS avg_amount
    FROM orders
    WHERE status = 'paid' AND quantity >= 2
    GROUP BY category
    ORDER BY avg_amount DESC, category
    LIMIT 3
""")
sql_rows = [tuple(r) for r in sql_df.collect()]
sql_rows
```

Result:

```
[('toys', 436.56), ('music', 425.46), ('sports', 420.1)]
```

</details>

<details>
<summary><b>2.3</b> &nbsp;Answer the same question with the DataFrame API (api_df, rows in api_rows).</summary>

```python
api_df = (orders.filter((F.col("status") == "paid") & (F.col("quantity") >= 2))
          .groupBy("category").agg(F.round(F.avg("amount"), 2).alias("avg_amount"))
          .orderBy(F.desc("avg_amount"), "category").limit(3))
api_rows = [tuple(r) for r in api_df.collect()]
same_plan = operators(sql_df) == operators(api_df)
api_rows, same_plan
```

Result:

```
([('toys', 436.56), ('music', 425.46), ('sports', 420.1)], True)
```

</details>

<details>
<summary><b>2.4</b> &nbsp;Read the four plans of q (q.explain("extended")) and answer from them:</summary>

```python
q = (orders.join(customers, "customer_id")
       .filter(F.col("amount") > 500)
       .groupBy("home_country").count())
q.explain("extended")

scan = [b for b in plan(q).split("\n\n") if b.strip().startswith("(") and "Scan parquet" in b.strip().splitlines()[0]][0]
pushed = re.search(r"PushedFilters: \[(.*)\]", scan).group(1).split(", ")
built = "customers"
n_shuffles = count_nodes(q, "Exchange")["Exchange"]
pushed, built, n_shuffles
```

Result:

```
== Parsed Logical Plan ==
'Aggregate ['home_country], ['home_country, 'count(1) AS count#87]
+- Filter (amount#24 > cast(500 as double))
   +- Project [customer_id#16L, order_id#15L, customer_name#17, country#18, segment#19, product_id#20, category#21, price#22, quantity#23, amount#24, status#25, payment_method#26, order_ts#27, order_date#28, tracking_code#29, home_country#34, customer_segment#35]
      +- Join Inner, (customer_id#16L = customer_id#33L)
         :- Relation [order_id#15L,customer_id#16L,customer_name#17,country#18,segment#19,product_id#20,category#21,price#22,quantity#23,amount#24,status#25,payment_method#26,order_ts#27,order_date#28,tracking_code#29] parquet
         +- Relation [customer_id#33L,home_country#34,customer_segment#35] parquet

== Analyzed Logical Plan ==
home_country: string, count: bigint
Aggregate [home_country#34], [home_country#34, count(1) AS count#87
...
```

</details>

<details>
<summary><b>2.5</b> &nbsp;Which Catalyst rules changed the plan of wasteful, and which rule removed the silly 1 + 1 == 2?</summary>

```python
wasteful = (orders.select("*")
            .join(customers, "customer_id")
            .filter((F.col("amount") > 500) & (F.lit(1) + F.lit(1) == 2))
            .select("home_country", "amount"))
fired = rules_fired(wasteful)
folding_rule = "ConstantFolding"
sorted(fired)
```

Result:

```
['BooleanSimplification',
 'CollapseProject',
 'ColumnPruning',
 'ConstantFolding',
 'InferFiltersFromConstraints',
 'PushDownPredicates',
 'RemoveNoopOperators',
 'ResolveReferences']
```

</details>

<details>
<summary><b>2.6</b> &nbsp;Turn off the rule PushDownPredicates (the setting spark.sql.optimizer.excludedRules takes the ...</summary>

```python
spark.conf.set("spark.sql.optimizer.excludedRules", "org.apache.spark.sql.catalyst.optimizer.PushDownPredicates")
wasteful2 = (orders.select("*")
             .join(customers, "customer_id")
             .filter((F.col("amount") > 500) & (F.lit(1) + F.lit(1) == 2))
             .select("home_country", "amount"))
print(optimized(wasteful2))
filter_above_join = True
spark.conf.unset("spark.sql.optimizer.excludedRules")
```

Result:

```
Project [home_country#34, amount#24]
+- Join Inner, (customer_id#16L = customer_id#33L)
   :- Filter (isnotnull(amount#24) AND (amount#24 > 500.0))
   :  +- Project [customer_id#16L, amount#24]
   :     +- Filter isnotnull(customer_id#16L)
   :        +- Relation [order_id#15L,customer_id#16L,customer_name#17,country#18,segment#19,product_id#20,category#21,price#22,quantity#23,amount#24,status#25,payment_method#26,order_ts#27,order_date#28,tracking_code#29] parquet
   +- Project [customer_id#33L, home_country#34]
      +- Filter isnotnull(customer_id#33L)
         +- Relation [customer_id#33L,home_country#34,customer_segment#35] parquet
```

</details>

<details>
<summary><b>2.7</b> &nbsp;When does Spark add a shuffle for a groupBy?</summary>

```python
exchanges = [
    count_nodes(orders.groupBy("customer_id").count(), "Exchange")["Exchange"],
    count_nodes(orders.repartition("customer_id").groupBy("customer_id").count(), "Exchange")["Exchange"],
    count_nodes(orders.repartition(8).groupBy("customer_id").count(), "Exchange")["Exchange"],
]
exchanges
```

Result:

```
[1, 1, 2]
```

</details>

<details>
<summary><b>2.8</b> &nbsp;Filter orders to amount over 500 twice: with a normal column condition and with a Python UDF.</summary>

```python
from pyspark.sql.functions import udf
over_500 = udf(lambda a: a is not None and a > 500, "boolean")
builtin_df = orders.filter(F.col("amount") > 500)
udf_df = orders.filter(over_500("amount"))

def pushed_of(df):
    m = re.search(r"PushedFilters: \[(.*?)\]", plan(df))
    return m.group(1).split(", ") if m and m.group(1) else []

pushed_builtin, pushed_udf = pushed_of(builtin_df), pushed_of(udf_df)
t_builtin, t_udf = timeit(builtin_df.count), timeit(udf_df.count)
pushed_builtin, pushed_udf, t_builtin, t_udf
```

Result:

```
(['IsNotNull(amount)', 'GreaterThan(amount,500.0)'], [], 0.14, 1.07)
```

</details>

<details>
<summary><b>2.9</b> &nbsp;Which columns does Spark read from the Parquet files for orders.agg(F.sum("amount") ...</summary>

```python
pruned = orders.agg(F.sum("amount"), F.sum("quantity"))
schema = re.search(r"ReadSchema: struct<(.*)>", plan(pruned)).group(1)
read_cols = sorted(c.split(":")[0] for c in schema.split(","))
read_cols
```

Result:

```
['amount', 'quantity']
```

</details>

<details>
<summary><b>2.10</b> &nbsp;How many whole-stage codegen regions (n) does the simple plan of cancel_q have, and which ...</summary>

```python
simple = plan(cancel_q, "simple")
n_codegen = len(set(re.findall(r"\*\((\d+)\)", simple)))
separator = "Exchange"
print(simple)
n_codegen, separator
```

Result:

```
== Physical Plan ==
*(3) Sort [payment_method#26 ASC NULLS FIRST], true, 0
+- Exchange rangepartitioning(payment_method#26 ASC NULLS FIRST, 200), ENSURE_REQUIREMENTS, [plan_id=106]
   +- *(2) HashAggregate(keys=[payment_method#26], functions=[count(1)])
      +- Exchange hashpartitioning(payment_method#26, 200), ENSURE_REQUIREMENTS, [plan_id=102]
         +- *(1) HashAggregate(keys=[payment_method#26], functions=[partial_count(1)])
            +- *(1) Project [payment_method#26]
               +- *(1) Filter (isnotnull(status#25) AND (status#25 = cancelled))
                  +- *(1) ColumnarToRow
                     +- FileScan parquet [status#25,payment_method#26] Batched: true, DataFilters: [isnotnull(status#25), (status#25 = cancelled)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/private/tmp/claude-501/-Users-saidakbar-Documents-innowise-cours..., PartitionFilters: 
...
```

</details>

<details>
<summary><b>2.11</b> &nbsp;How many orders were placed on each weekday (0 = Monday to 6 = Sunday)?</summary>

```python
rdd_job = lambda: dict(orders.rdd.map(lambda r: (r.order_date.weekday(), 1)).reduceByKey(lambda a, b: a + b).collect())
df_job = lambda: {r[0]: r[1] for r in orders.groupBy(F.weekday("order_date")).count().collect()}
t_rdd, t_df = timeit(rdd_job), timeit(df_job)
rdd_counts, df_counts = rdd_job(), df_job()
t_rdd, t_df
```

Result:

```
(74.98, 0.57)
```

</details>

<details>
<summary><b>2.12</b> &nbsp;Save orders as the table orders_t. How well does the optimizer estimate the number of orders ...</summary>

```python
orders.write.mode("overwrite").saveAsTable("orders_t")
over_900 = lambda: spark.table("orders_t").filter("amount > 900")

est_none = estimate(over_900())
spark.sql("ANALYZE TABLE orders_t COMPUTE STATISTICS FOR COLUMNS amount")
spark.conf.set("spark.sql.cbo.enabled", "true")
est_uniform = estimate(over_900())
spark.conf.set("spark.sql.statistics.histogram.enabled", "true")
spark.sql("ANALYZE TABLE orders_t COMPUTE STATISTICS FOR COLUMNS amount")
est_histogram = estimate(over_900())
actual = over_900().count()
est_none, est_uniform, est_histogram, actual
```

Result:

```
((56139026, None), (20111600, 100558), (9445000, 47225), 45561)
```

</details>

<details>
<summary><b>2.13</b> &nbsp;For every payment_method, report the number of orders (orders), the share of cancelled orders ...</summary>

```python
by_payment_df = (orders.groupBy("payment_method")
    .agg(F.count("*").alias("orders"),
         F.round(100 * F.sum(F.when(F.col("status") == "cancelled", 1).otherwise(0)) / F.count("*"), 1).alias("cancel_pct"),
         F.round(F.avg(F.when(F.col("status") == "paid", F.col("amount"))), 2).alias("avg_paid"))
    .orderBy("payment_method"))
by_payment = [tuple(r) for r in by_payment_df.collect()]
by_payment
```

Result:

```
[('card', 600405, 12.0, 370.57),
 ('cash', 249395, 12.0, 370.19),
 ('wallet', 150200, 11.8, 371.73)]
```

</details>

<details>
<summary><b>2.14</b> &nbsp;How many orders of each status were placed in each month?</summary>

```python
monthly_df = (orders.groupBy(F.month("order_date").alias("month"))
    .pivot("status", ["cancelled", "paid", "returned", "shipped"]).count().orderBy("month"))
monthly = [tuple(r) for r in monthly_df.collect()]
monthly
```

Result:

```
[(1, 41146, 187017, 27290, 85766),
 (2, 38022, 175164, 25335, 79988),
 (3, 40814, 186593, 27247, 85618)]
```

</details>

<details>
<summary><b>2.15</b> &nbsp;Which customers placed at least 100 orders?</summary>

```python
heavy = (orders.filter(F.col("customer_id").isNotNull())
         .groupBy("customer_id").agg(F.count("*").alias("orders"))
         .filter(F.col("orders") >= 100))
n_heavy = heavy.count()
top3 = [(r["customer_id"], r["orders"]) for r in heavy.orderBy(F.desc("orders"), "customer_id").limit(3).collect()]
n_heavy, top3
```

Result:

```
(3, [(1, 293049), (4662, 101), (6518, 101)])
```

</details>